<a href="https://colab.research.google.com/github/murugesan-boopathy/agentic_ai_day1/blob/main/Day1_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

What You Will Learn

* Hands-on: Develop an autonomous AI patent search assistant using Python, Ollama and LangGraph with structured outputs and tool execution.


The Problem We Are Solving

*   While submitting patents, Prior Art Search is a difficult and complex process which needs to be captured and submitted. This AI patent search assitant agent will try to fetch the existing patents based on keywords provided.


⏱ Expected time: 20 minutes

# New Section

In [ ]:
# Step 1: Switch to a GPU Runtime

# Ollama runs significantly faster using Google Colab's free T4 GPU.
# 1. Click on Runtime in the top menu.
# 2. Select Change runtime type.
# 3. Choose T4 GPU under Hardware Accelerator and click Save.

# Step 2: Download and Install Ollama
# Run this command in a new code cell to install the required system tools and the official Ollama Linux binary:
!sudo apt-get update
!sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/u

In [ ]:

# Step 3: Start the Ollama Server in the Background
# Because Colab cells execute sequentially, running ollama serve directly will hijack the cell and block you from executing further code. Use Python’s subprocess library to launch it safely in the background:

import subprocess
import time
import os

os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "4096"

# Writes the output to ollama.log file with write option
log_file = open("ollama.log", "w")

# Start the Ollama server in a background process
#ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

# Wait 10 seconds for the server to spin up completely
time.sleep(10)

# Step 4
# Pull the precise AI model and embedding weights
!ollama pull mistral
!ollama pull nomic-embed-text
print("✅ Ollama Background Engine is active and ready!")



✅ Ollama Background Engine is active and ready!


In [ ]:
# Check the logs during crash
!tail -n 30 ollama.log

time=2026-11-11T17:49:23.624Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:4096 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:true OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:1 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 http

In [ ]:
# Quick Health Check

# Before sending prompts, verify that Ollama is actively listening on port 11434 by running this command in a cell
!curl -s http://localhost:11434/api/tags || echo "Ollama is down"


{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-11-11T17:49:35.094872574Z","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16","context_length":2048,"embedding_length":768,"runner":"ggml"},"capabilities":["embedding"]},{"name":"mistral:latest","model":"mistral:latest","modified_at":"2026-11-11T17:49:34.577822445Z","size":4372824384,"digest":"6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"7.2B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":4096,"runner":"ggml"},"capabilities":["completion","tools"]}]}

In [ ]:

# Step 5: Test via Python API

# You can quickly check if the model is responding using a standard curl request inside Python:
import requests
import json

url = "http://localhost:11434/api/generate"
payload = {
    "model": "mistral",
    "prompt": "Why is the sky blue? Answer in one sentence.",
    "stream": False
}

response = requests.post(url, json=payload)
print(json.loads(response.text)['response'])


 The sky appears blue during the day because molecules in Earth's atmosphere scatter sunlight more strongly in the blue direction than other colors, a phenomenon known as Rayleigh scattering.


In [ ]:
# ==============================================================================
# STEP 6: Install Dependencies
# ==============================================================================

!pip install --force-reinstall \
  "opentelemetry-api==1.42.1" \
  "opentelemetry-sdk==1.42.1" \
  "google-adk==2.7.1" \
  "opentelemetry-exporter-otlp-proto-grpc==1.42.1" \
  "google-auth==2.49.0" \
  "requests==2.32.4" \
  "python-dateutil<3" \
  "oauthlib<4" \
  "pandas==2.2.3" \
  "numpy<2.5,>=1.22" \
  "fsspec==2025.12.0" \
  "huggingface-hub<2.0,>=0.25.0" \
  "rich<14,>=12.4.4" \
  langgraph \
  langchain-ollama \
  pydantic \
  langchain-chroma \
  openpyxl \
  beautifulsoup4 \
  jedi

  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached google_adk-2.7.1-py3-none-any.whl.metadata (24 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached google_auth-2.49.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
  Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached numpy-2.4.6-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached rich-13.9.4-py3-none-any.whl.metadata (18 kB

In [ ]:
# Core

# ==============================================================================
# STEP 2: Initialize Core Models & Local Infrastructure Vault
# ==============================================================================
import os
import json
import smtplib
import requests
import pandas as pd
from email.mime.text import MIMEText
from bs4 import BeautifulSoup
from typing import Annotated, Sequence, Dict, Any, List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# Initialize local embedding and LLM models via Ollama
embeddings = OllamaEmbeddings(model="nomic-embed-text", keep_alive=0)
llm = ChatOllama(model="mistral", temperature=0, num_ctx=4096, keep_alive="10m")

# Build a mock production DB designed to fail vector queries for text anomalies
DB_PATH = "./production_patent_vault"
os.makedirs(DB_PATH, exist_ok=True)

MOCK_VAULT_DOCS = [
    Document(page_content="Cryogenic error-correction matrix block stabilizing physical superconducting qubits.", metadata={"id": "US-11543210-B1", "title": "Quantum Error Correction Matrix", "cpc": "H01L", "year": 2024}),
    Document(page_content="Transformer model trained to parse legal claims and locate relevant prior art.", metadata={"id": "US-10984321-B2", "title": "Transformer-based Prior Art Parser", "cpc": "G06N", "year": 2019})
]
remote_vector_client = Chroma.from_documents(documents=MOCK_VAULT_DOCS, embedding=embeddings, persist_directory=DB_PATH)

# ==============================================================================
# STEP 3: Define Schemas & Internal Database Extraction Tools
# ==============================================================================
class AdvancedPatentSearchSchema(BaseModel):
    semantic_query: str = Field(description="The functional keywords or technology context to pull semantically.")
    cpc_classification: str = Field(description="Mandatory Cooperative Patent Classification standard code (e.g., G06N).")
    min_year: int = Field(default=2000, description="The oldest publication timeline limit.")

@tool(args_schema=AdvancedPatentSearchSchema)
def production_vector_patent_search(semantic_query: str, cpc_classification: str, min_year: int = 2000) -> List[Dict[str, Any]]:
    """Queries the internal database vault for matching structural patent records."""
    raw_hits = remote_vector_client.similarity_search(semantic_query, k=2)
    filtered_results = []
    for doc in raw_hits:
        meta = doc.metadata
        if meta["cpc"] == cpc_classification and meta["year"] >= min_year:
            filtered_results.append({
                "patent_id": meta["id"],
                "title": meta["title"],
                "cpc": meta["cpc"],
                "year": meta["year"],
                "abstract_snippet": doc.page_content,
                "source": "Internal DB Store"
            })
    return filtered_results if filtered_results else [{"message": "Zero records matched."}]

llm_with_tools = llm.bind_tools([production_vector_patent_search])

# Output Structural Targets
class PatentPatentReport(BaseModel):
    patent_id: str = Field(description="Unique regulatory alphanumeric code identifier.")
    technical_relevance: str = Field(description="Evaluation of cross-functional claim alignments.")
    source_origin: str = Field(description="Traces if data point came from 'Internal DB' or 'Google Patents Web Scraper'.")

class MasterPatentSearchResponse(BaseModel):
    executive_summary: str = Field(description="Comprehensive technical analysis summary.")
    matching_patents: List[PatentPatentReport] = Field(description="Array of validated patent targets passing quality checks.")
    recommended_next_steps: List[str] = Field(description="Strategic options checklist.")

structured_llm = llm.with_structured_output(MasterPatentSearchResponse)

# Quality Assessment Evaluation Schema
class RelevanceAssessmentSchema(BaseModel):
    relevance_score: int = Field(description="Score from 1 to 5 indicating how well raw database hits match user search parameters.")
    critique: str = Field(description="Brief critical assessment justifying the score assigned.")

evaluator_llm = llm.with_structured_output(RelevanceAssessmentSchema)

# ==============================================================================
# STEP 4: Define State Architecture & Advanced Node Tasks
# ==============================================================================
class ProductionAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    cpc_code: str
    is_data_validated: bool
    scraped_backup_data: str
    final_json_report: Dict[str, Any]

def input_validator_node(state: ProductionAgentState):
    """Parses context inputs to confirm strict CPC code validation constraints are met."""
    last_msg = state["messages"][-1].content.lower()
    detected_cpc = None
    for code in ["g06n", "h01l"]:
        if code in last_msg:
            detected_cpc = code.upper()
    if detected_cpc:
        return {"cpc_code": detected_cpc}
    if not state.get("cpc_code"):
        return {"cpc_code": "MISSING"}
    return {}

def orchestrator_node(state: ProductionAgentState):
    """Core AI processing agent node."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def human_clarification_node(state: ProductionAgentState):
    """Halts graph execution to ask the human agent for confirmation inputs."""
    return {"messages": [AIMessage(content="⚠️ System Halt: A specific CPC classification code (e.g., 'G06N' or 'H01L') is required to query the database.")]}

def rag_evaluation_node(state: ProductionAgentState):
    """Evaluates internal DB query text snippets to gauge target relevance thresholds."""
    tool_msg = state["messages"][-1].content
    user_query = state["messages"][0].content

    evaluation_prompt = f"Evaluate how relevant these internal DB results are to the user query. Query: {user_query}. Results: {tool_msg}"
    assessment = evaluator_llm.invoke([SystemMessage(content="You are a strict technical data auditor."), HumanMessage(content=evaluation_prompt)])

    print(f"\n🔬 [EVALUATION LOG]: Quality Score: {assessment.relevance_score}/5 | Reasoning: {assessment.critique}")
    return {"is_data_validated": assessment.relevance_score >= 3}

# ==============================================================================
# NEW STEP: Email Notification Node
# ==============================================================================
def email_notification_node(state: ProductionAgentState):
    """Dispatches a team alert when internal databases miss validation metrics."""
    cpc = state.get("cpc_code", "UNKNOWN")
    user_query = state["messages"][0].content

    subject = f"🚨 [PATENT SYSTEM ALERT] Fallback Scraper Triggered for CPC: {cpc}"
    body = f"""
    The Autonomous Patent Search Assistant has triggered an external fallback operation.

    Reason: Internal database records failed the semantic data validation threshold.
    Target Classification: {cpc}
    Original User Query: '{user_query}'

    The Google Patents Live Web Scraper is now executing to maintain pipeline integrity.
    """

    # Extract optional SMTP configuration fields from your workspace environment
    smtp_server = os.environ.get("SMTP_SERVER", "://gmail.com")
    smtp_port = int(os.environ.get("SMTP_PORT", "587"))
    sender_email = os.environ.get("SENDER_EMAIL")
    sender_password = os.environ.get("SENDER_PASSWORD")
    recipient_email = os.environ.get("RECIPIENT_EMAIL", "engineering_team@company.com")

    if sender_email and sender_password:
        try:
            msg = MIMEText(body)
            msg["Subject"] = subject
            msg["From"] = sender_email
            msg["To"] = recipient_email

            with smtplib.SMTP(smtp_server, smtp_port) as server:
                server.starttls()
                server.login(sender_email, sender_password)
                server.sendmail(sender_email, [recipient_email], msg.as_string())
            print(f"\n📧 [EMAIL DISPATCHED]: Secure alert notification successfully sent to {recipient_email}")
        except Exception as e:
            print(f"\n⚠️ [EMAIL SMTP ERROR]: Failed to route email natively: {str(e)}")
    else:
        # Resilient local fallback logging framework if run headless without setup variables
        print(f"\n📧 [MOCK EMAIL ALERT LOGGER]:")
        print(f"  To: {recipient_email}")
        print(f"  Subject: {subject}")
        print(f"  Content Summary: Internal data quality fell below threshold. Initializing external scrapers.")

    return {}


def google_patents_scraper_node(state: ProductionAgentState):
    """Falls back to public Google Patents indices if internal stores are insufficient."""
    print("\n🌐 [SCRAPER NODE]: Launching corrected Google Patents Web Scraper pipeline...")

    original_user_query = state["messages"][0].content
    cpc = state.get("cpc_code", "G06N")

    clean_query = original_user_query.split("under classification")[0].strip()
    encoded_query = requests.utils.quote(clean_query)

    # Raw string concatenation prevents the path slash from getting stripped
    search_url = "https://google.com" + "/?q=" + str(encoded_query) + "&cc=" + str(cpc) + "&num=2"

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

    # FIX: Explicitly initialized with empty brackets to fix the SyntaxError
    scraped_patents = []

    try:
        res = requests.get(search_url, headers=headers, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, "html.parser")
            meta_results = soup.find_all("article", class_="result-item")[:2]

            for index, item in enumerate(meta_results):
                title_el = item.find("span", id="title")
                num_el = item.find("span", id="patentNumber")
                snippet_el = item.find("div", class_="abstract")

                scraped_patents.append({
                    "patent_id": num_el.text.strip() if num_el else f"US-SCRA-00{index}-B2",
                    "technical_relevance": snippet_el.text.strip() if snippet_el else f"Prior art matching {clean_query} concepts.",
                    "source_origin": "Google Patents Scraper Fallback"
                })
        else:
            raise Exception(f"HTTP Return Code: {res.status_code}")

    except Exception as e:
        print(f"⚠️ Live web scraper blocked or timed out ({str(e)}). Running clean simulation safety fallback data...")
        scraped_patents = [
            {
                "patent_id": "US-12001122-B1",
                "technical_relevance": f"Automated fallback tracking matching {clean_query} configurations.",
                "source_origin": "Google Patents Scraper Fallback"
            }
        ]

    print(f"🌐 [SCRAPER SUCCESS]: Gathered {len(scraped_patents)} fallback elements from external data paths.")
    scraper_payload = AIMessage(content=f"Google Patents Scraper findings: {json.dumps(scraped_patents)}")
    return {"messages": [scraper_payload], "scraped_backup_data": json.dumps(scraped_patents)}


def output_formatter_node(state: ProductionAgentState):
  """Compiles system context logs into a strict JSON payload."""
  sys_prompt = SystemMessage(content="Compile the discovery logs into the requested MasterPatentSearchResponse JSON schema.")
  structured_report = structured_llm.invoke([sys_prompt] + list(state["messages"]))
  return {"final_json_report": structured_report.model_dump()}

def drive_export_node(state: ProductionAgentState):
  """Compiles spreadsheet data files stored locally or via Google Drive paths."""
  report_data = state.get("final_json_report", {})
  if "error" in report_data or not report_data:
    return {}
  export_dir = "/content/drive/MyDrive/Patent_Reports" if os.path.exists("/content/drive") else "./generated"
  os.makedirs(export_dir, exist_ok=True)
  file_name = f"{export_dir}/Patent_Report_{state.get('cpc_code', 'RAW')}.xlsx"
  df_patents = pd.DataFrame(report_data.get("matching_patents", []))
  with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
    df_patents.to_excel(writer, sheet_name="Matching Patents", index=False)
  pd.DataFrame([{"Executive Summary": report_data.get("executive_summary")}]).to_excel(writer, sheet_name="Summary", index=False)
  pd.DataFrame({"Next Steps": report_data.get("recommended_next_steps")}).to_excel(writer, sheet_name="Next Steps", index=False)
  print(f"💾 [EXPORT SUCCESS]: Saved multi-sheet patent tracker workbook to: {file_name}")
  return {}

# --- Conditional Edge Routing ---

def check_initial_input_compliance(state: ProductionAgentState):
  return "human_pause" if state.get("cpc_code") == "MISSING" else "agent"

def evaluate_agent_next_step(state: ProductionAgentState):
  return "tools" if getattr(state["messages"][-1], "tool_calls", None) else "evaluator"

def route_evaluation_results(state: ProductionAgentState):
  # Route to Email Node when validation fails; skip directly to formatter if internal results pass
  return "format_output" if state.get("is_data_validated") else "emailer"

# ==============================================================================


# STEP 5: Build State Machine Framework Graph Architecture


# ==============================================================================

builder = StateGraph(ProductionAgentState)
builder.add_node("validator", input_validator_node)
builder.add_node("agent", orchestrator_node)
builder.add_node("human_pause", human_clarification_node)
builder.add_node("tools", ToolNode([production_vector_patent_search]))
builder.add_node("evaluator", rag_evaluation_node)
builder.add_node("emailer", email_notification_node)  # Advanced alerting layer
builder.add_node("scraper", google_patents_scraper_node)
builder.add_node("format_output", output_formatter_node)
builder.add_node("drive_exporter", drive_export_node)
builder.add_edge(START, "validator")
builder.add_conditional_edges("validator", check_initial_input_compliance, {"human_pause": "human_pause", "agent": "agent"})
builder.add_edge("human_pause", END)
builder.add_conditional_edges("agent", evaluate_agent_next_step, {"tools": "tools", "evaluator": "evaluator"})
builder.add_edge("tools", "evaluator")
builder.add_conditional_edges("evaluator", route_evaluation_results, {"format_output": "format_output", "emailer": "emailer"})
builder.add_edge("emailer", "scraper")  # Chain Emailer sequentially directly into Scraper execution
builder.add_edge("scraper", "format_output")
builder.add_edge("format_output", "drive_exporter")
builder.add_edge("drive_exporter", END)
memory_layer = MemorySaver()
production_app = builder.compile(checkpointer=memory_layer)
#---

### Run Verification Test Case

# Execute the runtime cell below in your Google Colab environment to test the complete architecture:

# To test real live email routing, uncomment and populate the variables below:
# os.environ["SENDER_EMAIL"] = "your_email@gmail.com"
# os.environ["SENDER_PASSWORD"] = "your_app_password"
# os.environ["RECIPIENT_EMAIL"] = "target_team@company.com"

thread_config = {"configurable": {"thread_id": "email_alert_session_777"}}
target_query = {"messages": [HumanMessage(content="Search for high-speed computer vision systems or LiDAR array processors under classification G06N")]}

print("--- 🛰️ Launching Production Patent Engine Pipeline ---\n")

for event in production_app.stream(target_query, config=thread_config):
    for node, data in event.items():
        if data and "messages" in data and len(data["messages"]) > 0:
            print(f"[{node}]: Processing step complete.")

final_snapshot = production_app.get_state(thread_config)
print("\n🎯 FINAL STRUCTURED JSON REPORT WITH LOGGED METRICS:")
print(json.dumps(final_snapshot.values.get("final_json_report"), indent=2))

--- 🛰️ Launching Production Patent Engine Pipeline ---

[agent]: Processing step complete.
[tools]: Processing step complete.

🔬 [EVALUATION LOG]: Quality Score: 1/5 | Reasoning: The provided result is highly relevant to the user query as it falls under the same classification G06N and the patent title does not explicitly mention high-speed computer vision systems or LiDAR array processors, but the Transformer model could potentially be applied to these areas. However, the abstract snippet suggests the patent is about parsing legal claims and locating relevant prior art, not directly related to high-speed computer vision systems or LiDAR array processors.

📧 [MOCK EMAIL ALERT LOGGER]:
  To: engineering_team@company.com
  Subject: 🚨 [PATENT SYSTEM ALERT] Fallback Scraper Triggered for CPC: G06N
  Content Summary: Internal data quality fell below threshold. Initializing external scrapers.

🌐 [SCRAPER NODE]: Launching corrected Google Patents Web Scraper pipeline...
🌐 [SCRAPER SUCCESS]: G

In [ ]:
from google.colab import files
import os

# Define the local generated path where the drive_export_node dumped the file
file_path = "./generated/Patent_Report_G06N.xlsx"

if os.path.exists(file_path):
    print("📁 Patent report located successfully. Initiating browser download...")
    files.download(file_path)
else:
    print("❌ Report file not found. Double-check your graph pipeline execution for errors.")


📁 Patent report located successfully. Initiating browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q gradio plotly collections-extended


/tmp/ipykernel_5401/137318139.py:62: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="AI Patent Search Intelligence Station") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c90088ea60bd286f04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔬 [EVALUATION LOG]: Quality Score: 3/5 | Reasoning: The provided result is not directly relevant to the user query 'Lithium-Sulfur Battery under classification H01L'. The patent title 'Quantum Error Correction Matrix' and the abstract snippet 'Cryogenic error-correction matrix block stabilizing physical superconducting qubits' do not indicate any connection to Lithium-Sulfur Batteries or their classification under H01L. However, the CPC classification 'H01L' matches the user query, which may suggest a broader relevance to the field of electrical technology.
💾 [EXPORT SUCCESS]: Saved multi-sheet patent tracker workbook to: ./generated/Patent_Report_H01L.xlsx

🔬 [EVALUATION LOG]: Quality Score: 0/5 | Reasoning: The provided results are not relevant to the user query. The search results do not contain any data related to Lithium-Sulfur Battery under the specified classification H01L. It is recommended to re-run the search with a more specific query or to adjust the search parameters to i

In [ ]:
import os
import json
import time
import random
import re
from collections import Counter
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr
from langchain_core.messages import HumanMessage

# Ensure output export directory exists cleanly
EXPORT_DIR = "./generated"
os.makedirs(EXPORT_DIR, exist_ok=True)

# ==============================================================================
# DATA EXTRACTION & DYNAMIC VISUALIZATION BUILDER WITH HTML EXPORT LOGIC
# ==============================================================================
def run_patent_agent_ui(user_prompt, cpc_selection, thread_id):
    """Executes the agentic workflow, renders charts, and dumps them to standalone HTML files."""
    if not user_prompt.strip():
        return ("⚠️ Error: Please enter a technical search phrase.", None, None,
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                None, None, None, None)

    if not thread_id.strip():
        thread_id = f"session_{int(time.time())}"

    thread_config = {"configurable": {"thread_id": thread_id}}
    full_prompt = f"{user_prompt} under classification {cpc_selection}" if cpc_selection != "Auto-Detect / Missing" else user_prompt
    inputs = {"messages": [HumanMessage(content=full_prompt)]}

    log_stream = [f"🚀 Initializing Assistant Stream (Thread ID: {thread_id})..."]

    try:
        for event in production_app.stream(inputs, config=thread_config):
            for node, data in event.items():
                log_stream.append(f"⚙️ Node active: [{node}]")
                if node == "human_pause":
                    log_stream.append("🛑 SYSTEM WAITING: Valid CPC classification missing. Please select one below to resume.")
                elif node == "evaluator":
                    log_stream.append("🔬 Auditing dataset relevance profiles...")
                elif node == "scraper":
                    log_stream.append("🌐 System Fallback: Scraped public patent archives successfully.")
                elif node == "drive_exporter":
                    log_stream.append("💾 Compiling data matrices to an Excel document...")
    except Exception as e:
        log_stream.append(f"❌ Pipeline Exception: {str(e)}")

    final_state = production_app.get_state(thread_config)
    report_json = final_state.values.get("final_json_report", {})

    cpc_code_used = final_state.values.get("cpc_code", "RAW")
    file_path = f"{EXPORT_DIR}/Patent_Report_{cpc_code_used}.xlsx"
    download_link = file_path if os.path.exists(file_path) else None
    formatted_json_output = json.dumps(report_json, indent=2) if report_json and "error" not in report_json else "No structural records found."

    records = report_json.get("matching_patents", []) if report_json and isinstance(report_json, dict) else []

    if records and len(records) > 0:
        chart_data = []
        all_words = []

        STOPWORDS = set(["a", "an", "the", "and", "or", "but", "if", "then", "of", "at", "by", "for", "with", "about", "to", "in", "on", "is", "it", "that", "this", "from", "using", "system", "method", "apparatus", "framework", "configured", "optimized"])
        MOCK_ASSIGNEES = ["DeepMind Technologies", "IBM Innovation Corp", "Waymo LLC", "Superconducting Labs", "Quantum Horizon Group"]

        for p in records:
            patent_id = p.get("patent_id", "Unknown")
            relevance_snippet = p.get("technical_relevance", "A deep neural learning framework layer.")

            words = re.findall(r'\b[a-zA-Z]{4,}\b', relevance_snippet.lower())
            all_words.extend([w for w in words if w not in STOPWORDS])

            chart_data.append({
                "Patent ID": patent_id,
                "Publication Year": random.choice([2021, 2022, 2023, 2024, 2025, 2026]),
                "Source Origin": "Google Patents Scraper" if "Scraper" in report_json.get("executive_summary", "") else "Internal DB Store",
                "Assignee Company": random.choice(MOCK_ASSIGNEES),
                "Relevance Value": random.randint(3, 5)
            })

        df = pd.DataFrame(chart_data)

        # --- Chart 1 Generation & Export ---
        fig_timeline = px.bar(
            df.groupby(["Publication Year", "Source Origin"]).size().reset_index(name="Patent Count"),
            x="Publication Year", y="Patent Count", color="Source Origin",
            title="🎯 Distributed Prior Art Publication Timelines & Origins",
            barmode="group", template="plotly_white"
        )
        fig_timeline.update_xaxes(type='category')
        timeline_html_path = f"{EXPORT_DIR}/timeline_chart_{cpc_code_used}.html"
        fig_timeline.write_html(timeline_html_path, include_plotlyjs="cdn")

        # --- Chart 2 Generation & Export ---
        fig_origins = px.pie(
            df, names="Source Origin", title="📊 System Data Source Yield Matrix Allocation",
            hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel
        )
        origins_html_path = f"{EXPORT_DIR}/origins_chart_{cpc_code_used}.html"
        fig_origins.write_html(origins_html_path, include_plotlyjs="cdn")

        # --- Chart 3 Generation & Export ---
        pivot_df = df.groupby(["Assignee Company", "Publication Year"]).size().unstack(fill_value=0)
        fig_heatmap = px.imshow(
            pivot_df,
            labels=dict(x="Publication Year", y="Assignee Company", color="Patents Filed"),
            x=pivot_df.columns.astype(str), y=pivot_df.index,
            title="🔥 Competitive Landscape Density Heatmap",
            color_continuous_scale="Viridis", text_auto=True
        )
        heatmap_html_path = f"{EXPORT_DIR}/heatmap_chart_{cpc_code_used}.html"
        fig_heatmap.write_html(heatmap_html_path, include_plotlyjs="cdn")

        # --- Chart 4 Generation & Export ---
        word_counts = Counter(all_words).most_common(12)
        if not word_counts:
            word_counts = [("network", 5), ("learning", 4), ("processor", 3), ("quantum", 3)]
        df_words = pd.DataFrame(word_counts, columns=["Keyword", "Frequency"])

        fig_wordcloud = go.Figure()
        fig_wordcloud.add_trace(go.Scatter(
            x=[random.uniform(1, 10) for _ in range(len(df_words))],
            y=[random.uniform(1, 10) for _ in range(len(df_words))],
            text=df_words["Keyword"], mode="text",
            textfont=dict(size=df_words["Frequency"] * 8, color=px.colors.qualitative.Dark24[:len(df_words)]),
            hovertext=[f"Frequency: {f}" for f in df_words["Frequency"]], hoverinfo="text"
        ))
        fig_wordcloud.update_layout(
            title="☁️ Abstract Technical Key-Term Frequency Weight Cloud",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            template="plotly_white"
        )
        wordcloud_html_path = f"{EXPORT_DIR}/wordcloud_chart_{cpc_code_used}.html"
        fig_wordcloud.write_html(wordcloud_html_path, include_plotlyjs="cdn")

        return ("\n".join(log_stream), formatted_json_output, download_link,
                gr.update(value=fig_timeline, visible=True), gr.update(value=fig_origins, visible=True),
                gr.update(value=fig_heatmap, visible=True), gr.update(value=fig_wordcloud, visible=True),
                timeline_html_path, origins_html_path, heatmap_html_path, wordcloud_html_path)

    return ("\n".join(log_stream), formatted_json_output, download_link,
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            None, None, None, None)

# ==============================================================================
# GRADIO 6.0 COMPLIANT INTELLIGENCE APPLICATION LAYOUT
# ==============================================================================
with gr.Blocks(title="Patent Search Analytics Engine") as demo:
    gr.Markdown(
        """
        # 🛰️ Corporate Patent Intelligence Dashboard with Interactive HTML Export
        *Fully compliant with Gradio 6.0. Save any dynamic visualization down as a robust, client-ready standalone web layout file.*
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔍 Configurations")
            user_input = gr.Textbox(label="Technical Query Area", placeholder="Type technical subject area here...", lines=3)
            cpc_dropdown = gr.Dropdown(choices=["Auto-Detect / Missing", "G06N", "H01L"], value="Auto-Detect / Missing", label="CPC Classification Filter")
            session_id = gr.Textbox(label="Conversation Session Token ID", value="html_export_vault_1")
            submit_btn = gr.Button("⚡ Trigger Agent Intelligence Sweep", variant="primary")
            gr.Markdown("---")
            file_output = gr.File(label="Download Excel Spreadsheet Report (.xlsx)")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("📋 System Discovery Matrix"):
                    execution_logs = gr.Textbox(label="Real-Time Node Stream Logs", lines=5, interactive=False)
                    json_display = gr.Code(label="Verified Structural Final JSON Output", language="json")

                with gr.TabItem("📈 Basic Trend Metrics"):
                    gr.Markdown("### 📊 Distribution Matrices")
                    with gr.Row():
                        with gr.Column():
                            timeline_chart = gr.Plot(label="Publication Timelines", visible=False)
                            dl_timeline = gr.File(label="📥 Download Timeline Chart (.html)")
                        with gr.Column():
                            origins_chart = gr.Plot(label="Data Yield Origins", visible=False)
                            dl_origins = gr.File(label="📥 Download Origins Chart (.html)")
                        with gr.TabItem("🏢 Corporate Competitive Analytics"):
                            gr.Markdown("### 🔬 Strategic Competitor Market Visualizations")
                        with gr.Row():
                            with gr.Column():
                                heatmap_chart = gr.Plot(label="Competitive Density Matrix Heatmap", visible=False)
                                dl_heatmap = gr.File(label="📥 Download Heatmap Matrix (.html)")
                            with gr.Column():
                                wordcloud_chart = gr.Plot(label="Key-Term Frequency Cloud Vector", visible=False)
                                dl_wordcloud = gr.File(label="📥 Download Word Cloud Vector (.html)")

# Explicit input-output component connection mapping

    submit_btn.click(
        fn=run_patent_agent_ui,
        inputs=[user_input, cpc_dropdown, session_id],
        outputs=[
            execution_logs, json_display, file_output,
            timeline_chart, origins_chart, heatmap_chart, wordcloud_chart,
            dl_timeline, dl_origins, dl_heatmap, dl_wordcloud
        ]
    )

# Clean, warning-free dashboard startup configuration passing theme parameters safely inside launch method hooks
demo.launch(debug=True, share=True, theme=gr.themes.Soft())

"""
def ensure_ollama():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
    except Exception:
        print("Restarting Ollama...")
        log_file = open("ollama.log", "a")
        subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
        time.sleep(5)
"""

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9e3f59c54776068294.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔬 [EVALUATION LOG]: Quality Score: 9/5 | Reasoning: The results are highly relevant to the user query as they are searching for Lithium-Sulfur Battery under the classification H01L. The 'production_vector_patent_search' result with the semantic query matching 'Lithium-Sulfur Battery' and the specified CPC classification 'H01L' indicates a direct match to the user's search intent.
💾 [EXPORT SUCCESS]: Saved multi-sheet patent tracker workbook to: ./generated/Patent_Report_H01L.xlsx

🔬 [EVALUATION LOG]: Quality Score: 0/5 | Reasoning: The provided results are not relevant to the user query as they are classified under G06N instead of H01L. The user query is for Lithium-Sulfur Battery under classification H01L, while the results are from a patent search with a CPC classification of G06N which pertains to 'Computing; Calculation; Counting'. The results do not contain any information about Lithium-Sulfur Batteries.

📧 [MOCK EMAIL ALERT LOGGER]:
  To: engineering_team@company.com
  Subject: 

'\ndef ensure_ollama():\n    try:\n        requests.get("http://localhost:11434/api/tags", timeout=2)\n    except Exception:\n        print("Restarting Ollama...")\n        log_file = open("ollama.log", "a")\n        subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)\n        time.sleep(5)\n'